# Development of the MLP architecture

## Pseudo Code

Torch stuff which I do not remember but I will check on the classes

---

1. Prepare our data to do this (I do not belive that we have this now) We need to break it into 3 size proteins. **DONE**
    - Avoid using loops, use built in vectorization
    - Window_size inside the MLP torch module structure
---

2. We need to have a lookup table with the same dimension as our vocab. In each iteration we go search that one and imporve it. Number of dimensions in here is optimizable as well)

    - padding -> window -> embeding -> architecture
    - emebding is a lookup table with a hyperparameter dimension

--- 

3. Now we concatenate them! Let's do this yee

---

4. Architecture to define in a standart normal neural network and the label to predict is the label at the end.

    - They suggest using tanh
    - We can decide on a certain architecture that we beleive is the best (How should we do this try out different ones?)

--- 
5. At the end a softmax because the output should be all possible tokens

    - At the end be careful with the one hot encoding
    - We need to look at nn.CrossEntropy something like this

## Imports and seed

In [80]:
from protgen.data import ProteinDataset
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens
from protgen.utils import set_seed
import torch
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [81]:
set_seed()

## Data

In [82]:
# Constants for the dataset and model
FAMILIES = ("AcyP", "HPr", "HU")
AA = "ACDEFGHIKLMNPQRSTVWY"
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]
total_tokens = list(special_tokens) + list(AA)
IGNORE_INDEX = -1  # what F.cross_entropy(ignore_index=...) must be given for the generative task

In [83]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

### Look at the data

In [11]:
X_train

tensor([[ 1, 13, 10,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13,  3,  ...,  0,  0,  0]])

In [12]:
y_train_labels 

tensor([1, 2, 1,  ..., 2, 2, 1])

In [13]:
y_train_shifted

tensor([[13, 10, 11,  ..., -1, -1, -1],
        [13, 14, 11,  ..., -1, -1, -1],
        [13, 19, 16,  ..., -1, -1, -1],
        ...,
        [13, 14, 11,  ..., -1, -1, -1],
        [13, 19, 12,  ..., -1, -1, -1],
        [13,  3,  6,  ..., -1, -1, -1]])

In [14]:
train_data.vocab

{'<PAD>': 0,
 '<BOS>': 1,
 '<EOS>': 2,
 'A': 3,
 'C': 4,
 'D': 5,
 'E': 6,
 'F': 7,
 'G': 8,
 'H': 9,
 'I': 10,
 'K': 11,
 'L': 12,
 'M': 13,
 'N': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'V': 20,
 'W': 21,
 'Y': 22}

In [15]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [16]:
train_data.tokens

tensor([[ 1, 13, 10,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13,  3,  ...,  0,  0,  0]])

## Test Data example

In [131]:
example_protein_seq_data = train_data[0][0]

In [132]:
example_protein_seq_data

tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])

## Model

In [133]:
from protgen.models.mlp import MLPLM

In [134]:
test_model = MLPLM(vocab_size = len(train_data.vocab),
                   context=3, 
                   stride = 1,
                   d_emb=32, 
                   d_hidden=256, 
                   pad_id=0)

In [135]:
test_model.forward(example_protein_seq_data, 
                   debug = True)

tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
Lenght of the sequence: 129
tensor([ 0,  0,  1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3,
        17, 15,  3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18,
        12,  6, 22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,
         5, 10, 13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,
         8, 10,  5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22,

tensor([[-0.0919, -0.0071,  0.0524,  ...,  0.3685,  0.1628,  0.0358],
        [-0.1868, -0.0276, -0.1071,  ...,  0.1028,  0.0611,  0.0739],
        [-0.0514,  0.2123, -0.3990,  ..., -0.2734,  0.2682,  0.1871],
        ...,
        [ 0.0148,  0.0233,  0.0029,  ...,  0.0265,  0.0749, -0.0490],
        [ 0.0148,  0.0233,  0.0029,  ...,  0.0265,  0.0749, -0.0490],
        [ 0.0148,  0.0233,  0.0029,  ...,  0.0265,  0.0749, -0.0490]],
       grad_fn=<AddmmBackward0>)

In [114]:
x = torch.arange(1., 8)
x.unfold(0, 2, 1)

tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.]])

In [115]:
t = torch.tensor([[[1,1,1],[2, 2, 2],[3, 3, 3]], [[2, 2, 2], [3, 3, 3], [4, 4, 4]]])
print(t)
torch.flatten(t,1)

tensor([[[1, 1, 1],
         [2, 2, 2],
         [3, 3, 3]],

        [[2, 2, 2],
         [3, 3, 3],
         [4, 4, 4]]])


tensor([[1, 1, 1, 2, 2, 2, 3, 3, 3],
        [2, 2, 2, 3, 3, 3, 4, 4, 4]])